In [32]:
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split , cross_val_score, StratifiedKFold , RandomizedSearchCV , GridSearchCV
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder , StandardScaler , PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

In [2]:
addiction_df = pd.read_csv("F://Smartphone_Addiction//Data//train.csv")

In [ ]:
report = sv.analyze(addiction_df)
report.show_html("F://Smartphone_Addiction//Reports//reports.html" , open_browser=False)

In [7]:
addiction_df.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [9]:
addiction_df.describe()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,addicted_label
count,691369.000000,662440.000000,595515.000000,557374.000000,564548.000000,639851.000000,646889.000000,623785.000000,610659.000000,579306.000000,691369.000000
mean,345684.000000,26.615408,7.640865,2.471038,1.459265,2.366971,6.804334,145.894900,102.636781,9.479866,0.709424
std,199581.183467,5.153162,2.721446,1.316137,0.934552,1.258797,1.234512,65.917556,48.093970,2.856006,0.454028
min,0.000000,18.000000,0.500000,0.000000,0.000000,0.000000,4.500000,20.000000,15.000000,0.510000,0.000000
25%,172842.000000,22.000000,5.480000,1.450000,0.700000,1.360000,5.780000,93.000000,64.000000,7.280000,0.000000
50%,345684.000000,27.000000,7.770000,2.310000,1.330000,2.200000,6.800000,150.000000,104.000000,9.580000,1.000000
75%,518526.000000,31.000000,9.840000,3.370000,2.090000,3.200000,7.870000,204.000000,145.000000,11.750000,1.000000
max,691368.000000,35.000000,15.000000,8.000000,4.000000,6.000000,9.000000,250.000000,180.000000,17.560000,1.000000


In [10]:
addiction_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       691369 non-null  int64  
 1   age                      662440 non-null  float64
 2   daily_screen_time_hours  595515 non-null  float64
 3   social_media_hours       557374 non-null  float64
 4   gaming_hours             564548 non-null  float64
 5   work_study_hours         639851 non-null  float64
 6   sleep_hours              646889 non-null  float64
 7   notifications_per_day    623785 non-null  float64
 8   app_opens_per_day        610659 non-null  float64
 9   weekend_screen_time      579306 non-null  float64
 10  gender                   662335 non-null  str    
 11  stress_level             636221 non-null  str    
 12  academic_work_impact     647145 non-null  str    
 13  addicted_label           691369 non-null  int64  
dtypes: float64(9), 

In [12]:
temp_addiction_df = addiction_df.copy()

In [13]:
temp_addiction_df.head(1)

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1


In [14]:
temp_addiction_df["age"] = temp_addiction_df["age"].fillna(temp_addiction_df["age"].median())

In [15]:
temp_addiction_df["sleep_hours"] = temp_addiction_df["sleep_hours"].fillna(temp_addiction_df["sleep_hours"].median())

In [16]:
temp_addiction_df["work_study_hours"] = temp_addiction_df["work_study_hours"].fillna(temp_addiction_df["work_study_hours"].median())

In [17]:
temp_addiction_df["social_media_hours"] = temp_addiction_df["social_media_hours"].fillna(0)

In [18]:
temp_addiction_df["gaming_hours"] = temp_addiction_df["gaming_hours"].fillna(0)

In [21]:
temp_addiction_df["daily_screen_time_hours"] = temp_addiction_df["daily_screen_time_hours"].fillna(temp_addiction_df["daily_screen_time_hours"].median())

In [22]:
temp_addiction_df["notifications_per_day"] = temp_addiction_df["notifications_per_day"].fillna(temp_addiction_df["notifications_per_day"].median())

In [23]:
temp_addiction_df["app_opens_per_day"] = temp_addiction_df["app_opens_per_day"].fillna(temp_addiction_df["app_opens_per_day"].median())

In [24]:
temp_addiction_df["weekend_screen_time"] = temp_addiction_df["weekend_screen_time"].fillna(temp_addiction_df["weekend_screen_time"].median())

In [27]:
cat_cols = ["gender", "stress_level", "academic_work_impact"]
cat_imputer = SimpleImputer(strategy='most_frequent')
temp_addiction_df[cat_cols] = cat_imputer.fit_transform(temp_addiction_df[cat_cols])

In [28]:
temp_addiction_df = pd.get_dummies(
    temp_addiction_df,
    columns=["gender", "stress_level", "academic_work_impact"],
    drop_first=True
)

In [ ]:
X = temp_addiction_df.drop(columns=["addicted_label"])
y = temp_addiction_df["addicted_label"]

In [33]:
X_sample, _, y_sample, _ = train_test_split(
    X, y,
    train_size=0.15,  
    stratify=y,
    random_state=42
)

In [41]:
X_sample = X_sample.drop(columns=['id'])

In [42]:
X_train, X_val, y_train, y_val = train_test_split(
    X_sample, y_sample,
    test_size=0.2,
    stratify=y_sample,
    random_state=42
)

In [43]:
model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42
)

In [44]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    model, X_sample, y_sample,
    cv=cv,
    scoring='roc_auc',
    n_jobs= 2
)

In [45]:
print(f"AUC per fold: {scores}")
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

AUC per fold: [0.95300371 0.95564948 0.95440042 0.95389134 0.95487726]
Mean AUC: 0.9544 (+/- 0.0009)


In [46]:
model.fit(X_sample, y_sample)
importances = pd.Series(model.feature_importances_, index=X_sample.columns).sort_values(ascending=False)
print(importances)

daily_screen_time_hours     0.459007
social_media_hours          0.164429
weekend_screen_time         0.129581
app_opens_per_day           0.048534
notifications_per_day       0.048161
work_study_hours            0.029406
gaming_hours                0.028001
age                         0.016764
sleep_hours                 0.016667
gender_Male                 0.013331
stress_level_Medium         0.012832
stress_level_Low            0.012148
gender_Other                0.010970
academic_work_impact_Yes    0.010170
dtype: float32


In [3]:
feature_engineering_data = addiction_df.copy()

In [6]:
feature_engineering_data["age"] = feature_engineering_data["age"].fillna(feature_engineering_data["age"].median())
feature_engineering_data["sleep_hours"] = feature_engineering_data["sleep_hours"].fillna(feature_engineering_data["sleep_hours"].median())
feature_engineering_data["work_study_hours"] = feature_engineering_data["work_study_hours"].fillna(feature_engineering_data["work_study_hours"].median())
feature_engineering_data["social_media_hours"] = feature_engineering_data["social_media_hours"].fillna(0)
feature_engineering_data["gaming_hours"] = feature_engineering_data["gaming_hours"].fillna(0)
feature_engineering_data["daily_screen_time_hours"] = feature_engineering_data["daily_screen_time_hours"].fillna(feature_engineering_data["daily_screen_time_hours"].median())
feature_engineering_data["notifications_per_day"] = feature_engineering_data["notifications_per_day"].fillna(feature_engineering_data["notifications_per_day"].median())
feature_engineering_data["app_opens_per_day"] = feature_engineering_data["app_opens_per_day"].fillna(feature_engineering_data["app_opens_per_day"].median())
feature_engineering_data["weekend_screen_time"] = feature_engineering_data["weekend_screen_time"].fillna(feature_engineering_data["weekend_screen_time"].median())
feature_engineering_data["total_time_spend"] = feature_engineering_data["daily_screen_time_hours"] + feature_engineering_data["weekend_screen_time"]
feature_engineering_data["social_media_share"] = feature_engineering_data["social_media_hours"] / feature_engineering_data["daily_screen_time_hours"].replace(0, np.nan)
feature_engineering_data["gaming_share"] = feature_engineering_data["gaming_hours"] / feature_engineering_data["daily_screen_time_hours"].replace(0, np.nan)
feature_engineering_data["avg_session_length"] = feature_engineering_data["daily_screen_time_hours"] / feature_engineering_data["app_opens_per_day"].replace(0, np.nan)
feature_engineering_data["notifications_per_open"] = feature_engineering_data["notifications_per_day"] / feature_engineering_data["app_opens_per_day"].replace(0, np.nan)
feature_engineering_data["weekend_vs_daily_ratio"] = feature_engineering_data["weekend_screen_time"] / feature_engineering_data["daily_screen_time_hours"].replace(0, np.nan)
feature_engineering_data["screen_to_sleep_ratio"] = feature_engineering_data["daily_screen_time_hours"] / feature_engineering_data["sleep_hours"].replace(0, np.nan)

In [9]:
cat_cols = ["gender", "stress_level", "academic_work_impact"]
cat_imputer = SimpleImputer(strategy='most_frequent')
feature_engineering_data[cat_cols] = cat_imputer.fit_transform(feature_engineering_data[cat_cols])

feature_engineering_data = pd.get_dummies(
    feature_engineering_data,
    columns=["gender", "stress_level", "academic_work_impact"],
    drop_first=True
)

In [10]:
X = feature_engineering_data.drop(columns=["addicted_label"])
y = feature_engineering_data["addicted_label"]

X_sample, _, y_sample, _ = train_test_split(
    X, y,
    train_size=0.15,
    stratify=y,
    random_state=42
)

X_sample = X_sample.drop(columns=['id'])

X_train, X_val, y_train, y_val = train_test_split(
    X_sample, y_sample,
    test_size=0.2,
    stratify=y_sample,
    random_state=42
)

model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    model, X_sample, y_sample,
    cv=cv,
    scoring='roc_auc',
    n_jobs=2
)

print(f"AUC per fold: {scores}")
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

AUC per fold: [0.95321448 0.95562764 0.95458265 0.95447598 0.95494932]
Mean AUC: 0.9546 (+/- 0.0008)


In [11]:
model.fit(X_sample, y_sample)
importances = pd.Series(model.feature_importances_, index=X_sample.columns).sort_values(ascending=False)
print(importances)

total_time_spend            0.597957
social_media_hours          0.090202
social_media_share          0.075962
app_opens_per_day           0.032827
notifications_per_day       0.029591
daily_screen_time_hours     0.022305
gaming_share                0.020548
weekend_screen_time         0.016957
work_study_hours            0.015056
gaming_hours                0.010590
age                         0.009455
sleep_hours                 0.009443
weekend_vs_daily_ratio      0.009174
gender_Male                 0.008558
screen_to_sleep_ratio       0.008405
notifications_per_open      0.008327
avg_session_length          0.007845
academic_work_impact_Yes    0.007468
gender_Other                0.006980
stress_level_Medium         0.006320
stress_level_Low            0.006027
dtype: float32


In [3]:
addiction_df.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [3]:
numeric_coloumns_median = ["age" , "daily_screen_time_hours" , "sleep_hours" , "notifications_per_day" , "app_opens_per_day" , "weekend_screen_time" , "work_study_hours"]
numeric_column_zero = ["social_media_hours" , "gaming_hours"]

In [ ]:
categorical_columns = ["gender", "stress_level", "academic_work_impact"]

In [5]:
numeric_pipeline_1 = Pipeline(
    [
        ("Imputer" , SimpleImputer(strategy="median")),
        ("Scaler" , StandardScaler())
    ]
)

In [6]:
numeric_pipeline_2 = Pipeline(
    [
        ("Imputer" , SimpleImputer(strategy='constant', fill_value=0)),
        ("Scaler" , StandardScaler())
    ]
)

In [7]:
categorical_pipeline = Pipeline(
    [
        ("Imputer" , SimpleImputer(strategy='most_frequent')),
        ("Encoder" , OneHotEncoder(handle_unknown="ignore"))        
    ]
)

In [8]:
preprocessor = ColumnTransformer(
    [
        ("num1" , numeric_pipeline_1 , numeric_coloumns_median),
        ("num2" , numeric_pipeline_2 , numeric_column_zero),
        ("categorical" , categorical_pipeline , categorical_columns)
    ]
)

In [9]:
final_df = addiction_df.copy()
X = final_df.drop(columns=["addicted_label"])
y = final_df["addicted_label"]

In [10]:
X_sample, _, y_sample, _ = train_test_split(
    X, y,
    train_size=0.15,  
    stratify=y,
    random_state=42
)

In [11]:
logistic_pipeline = Pipeline(
    [
        ("preprocessing" , preprocessor),
        ("model" , LogisticRegression(C=1.93 , max_iter=1000) )
    ]
)

In [21]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    logistic_pipeline, X_sample, y_sample,
    cv=cv,
    scoring='roc_auc',
    n_jobs=2
)

print(f"AUC per fold: {scores}")
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

AUC per fold: [0.90217386 0.90339265 0.90207576 0.90054219 0.90447052]
Mean AUC: 0.9025 (+/- 0.0013)


In [12]:
Svc_pipeline = Pipeline(
    [
        ("preprocessing" , preprocessor),
        ("model" , LinearSVC(C=100 , random_state=42) )
    ]
)

In [27]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    Svc_pipeline, X_sample, y_sample,
    cv=cv,
    scoring='roc_auc',
    n_jobs=2
)

print(f"AUC per fold: {scores}")
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

AUC per fold: [0.9026188  0.90368287 0.90229143 0.90078727 0.90476935]
Mean AUC: 0.9028 (+/- 0.0013)


In [25]:
Svc_poly_pipeline = Pipeline(
    [
        ("preprocessing", preprocessor),
        ("PolynomialFeatures", PolynomialFeatures(degree=2)),
        ("model", CalibratedClassifierCV(
            LinearSVC(tol=0.0001, max_iter=5000, loss='squared_hinge', C=10, random_state=42),
            cv=3
        ))
    ]
)

In [37]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    Svc_poly_pipeline, X_sample, y_sample,
    cv=cv,
    scoring='roc_auc',
    n_jobs=2
)

print(f"AUC per fold: {scores}")
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

AUC per fold: [0.92312498 0.92614377 0.92296528 0.92305243 0.92625684]
Mean AUC: 0.9243 (+/- 0.0015)


In [14]:
Tree_pipeline = Pipeline(
    [
        ("preprocessing" , preprocessor),
        ("model" , DecisionTreeClassifier(min_samples_split = 10, min_samples_leaf = 20, max_depth = 10, criterion = 'entropy' , random_state=42) )
    ]
)

In [40]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    Tree_pipeline, X_sample, y_sample,
    cv=cv,
    scoring='roc_auc',
    n_jobs=2
)

print(f"AUC per fold: {scores}")
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

AUC per fold: [0.92561204 0.92567295 0.92450632 0.92408248 0.92650795]
Mean AUC: 0.9253 (+/- 0.0009)


In [15]:
forest_pipeline = Pipeline(
    [
        ("preprocessing" , preprocessor),
        ("model" ,  RandomForestClassifier(n_estimators = 500, min_samples_split = 20, min_samples_leaf = 2, max_features = 'log2', max_depth = 20, criterion =  'entropy', random_state=42) )
    ]
)

In [30]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    forest_pipeline, X_sample, y_sample,
    cv=cv,
    scoring='roc_auc',
    n_jobs=2
)

print(f"AUC per fold: {scores}")
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

AUC per fold: [0.93865424 0.9399673  0.93790618 0.93807144 0.93961639]
Mean AUC: 0.9388 (+/- 0.0008)


In [16]:
boosted_forest_pipeline = Pipeline(
    [
        ("Preprocessing" , preprocessor),
        ("Model" , XGBClassifier(subsample = 0.8, n_estimators = 200, min_child_weight = 15, max_depth = 5, learning_rate = 0.1, gamma = 2, colsample_bytree = 0.8 ,objective='binary:logistic', eval_metric='auc', random_state=42))
    ]
)

In [25]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    boosted_forest_pipeline, X_sample, y_sample,
    cv=cv,
    scoring='roc_auc',
    n_jobs=2
)

print(f"AUC per fold: {scores}")
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

AUC per fold: [0.95142503 0.95289054 0.95148448 0.9516995  0.95290439]
Mean AUC: 0.9521 (+/- 0.0007)


In [22]:
param_dist = {
    "Model__n_estimators": [100, 150, 200],
    "Model__max_depth": [2, 3, 4, 5],       
    "Model__learning_rate": [0.01, 0.05, 0.1],
    "Model__subsample": [0.6, 0.7, 0.8],
    "Model__colsample_bytree": [0.6, 0.7, 0.8],
    "Model__gamma": [0.5, 1, 2, 5],           
    "Model__min_child_weight": [5, 7, 10, 15] 
}

In [23]:
random_boosted_search = RandomizedSearchCV(
    boosted_forest_pipeline,
    param_distributions=param_dist,
    n_iter=75,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=2
)

random_boosted_search.fit( X_sample, y_sample)
print(random_boosted_search.best_params_)
print(random_boosted_search.best_score_)

{'Model__subsample': 0.8, 'Model__n_estimators': 200, 'Model__min_child_weight': 15, 'Model__max_depth': 5, 'Model__learning_rate': 0.1, 'Model__gamma': 2, 'Model__colsample_bytree': 0.8}
0.9519509618806612


In [26]:
param_dist = {
    "model__n_estimators" : [50,100,200,300,400,500],
    "model__max_depth": [3, 5, 10, 15, 20, 50 , None],
    "model__max_features": ["sqrt" , "log2"],
    "model__min_samples_split": [2, 5, 10, 15, 20, 50],
    "model__min_samples_leaf": [1, 2, 5, 10, 20],
    "model__criterion": ["entropy", "gini", "log_loss"],
}

In [28]:
random_forest_search = RandomizedSearchCV(
    forest_pipeline,
    param_distributions=param_dist,
    n_iter=20,             
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=2              
)

random_forest_search.fit( X_sample, y_sample)
print(random_forest_search.best_params_)
print(random_forest_search.best_score_)

{'model__n_estimators': 500, 'model__min_samples_split': 20, 'model__min_samples_leaf': 2, 'model__max_features': 'log2', 'model__max_depth': 20, 'model__criterion': 'entropy'}
0.9388535584354727


In [31]:
param_dist = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "model__loss": ["hinge", "squared_hinge"],
    "model__class_weight": [None, "balanced"],
    "model__tol": [1e-4, 1e-3, 1e-2],
    "model__max_iter": [1000, 2000, 5000],
}

In [32]:
random_Svc_search = RandomizedSearchCV(
    Svc_poly_pipeline,
    param_distributions=param_dist,
    n_iter=20,             
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=2              
)

random_Svc_search.fit( X_sample, y_sample)
print(random_Svc_search.best_params_)
print(random_Svc_search.best_score_)

{'model__tol': 0.0001, 'model__max_iter': 5000, 'model__loss': 'squared_hinge', 'model__class_weight': None, 'model__C': 10}
0.9242759167553629


In [26]:
vc = VotingClassifier(
    estimators= [("poly_svc" , Svc_poly_pipeline) , ("forest" ,  forest_pipeline) , ("Boosted_forest" , boosted_forest_pipeline )],
    voting="soft"
)

In [27]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    vc, X_sample, y_sample,
    cv=cv,
    scoring='roc_auc',
    n_jobs=2
)

print(f"AUC per fold: {scores}")
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

AUC per fold: [0.94232317 0.9443565  0.94186766 0.9422782  0.9442888 ]
Mean AUC: 0.9430 (+/- 0.0011)


In [34]:
vc2 = VotingClassifier(
    estimators= [("poly_svc" , Svc_poly_pipeline) , ("forest" ,  forest_pipeline) , ("Boosted_forest" , boosted_forest_pipeline )],
    voting="soft",
    weights=[1,1,2]
)

In [30]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    vc2, X_sample, y_sample,
    cv=cv,
    scoring='roc_auc',
    n_jobs=2
)

print(f"AUC per fold: {scores}")
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

AUC per fold: [0.94232317 0.9443565  0.94186766 0.9422782  0.9442888 ]
Mean AUC: 0.9430 (+/- 0.0011)


In [31]:
param_grid = {'weights': [[1,1,1], [2,1,1], [1,2,1], [1,1,2], [3,1,1], [1,3,1], [2,2,1]] ,  "voting" : ["soft" , "hard"]}

In [33]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=vc,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=2,
    verbose=1
)

grid_search.fit(X_sample, y_sample)

print(f"Best AUC: {grid_search.best_score_:.4f}")
print(f"Best params: {grid_search.best_params_}")

Fitting 5 folds for each of 14 candidates, totalling 70 fits


f:\Smartphone_Addiction\.venv\Lib\site-packages\sklearn\model_selection\_search.py:1234: UserWarning: One or more of the test scores are non-finite: [0.94302287 0.93951576 0.94240533 0.94630013 0.93706336 0.94189736
 0.93987564        nan        nan        nan        nan        nan
        nan        nan]
  warnings.warn(


Best AUC: 0.9463
Best params: {'voting': 'soft', 'weights': [1, 1, 2]}


In [35]:
boosted_forest_pipeline.fit(X , y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Preprocessing', ...), ('Model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['id','age','daily_screen_time_hours',...,'gender','stress_level', 'academic_work_impact']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num1', ...), ('num2', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``rema

In [36]:
testing_df = pd.read_csv("F://Smartphone_Addiction//Data//test.csv")
id = testing_df["id"]  
df = testing_df.drop(columns=["id"])  

In [37]:
final_prediction = boosted_forest_pipeline.predict_proba(df)[:, 1]
submission = pd.DataFrame({
    "id" : id,
    "addicted_label": final_prediction
})

submission.to_csv("F://Smartphone_Addiction//predictions.csv" , index=False)